[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_14/NB_bishop_ch14_sampling.ipynb)

# Chapter 14 -- Sampling Methods

This notebook implements three fundamental Monte Carlo sampling methods:
- Rejection sampling
- Metropolis-Hastings algorithm
- Gibbs sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)
print(f'NumPy version: {np.__version__}')

## 1. Rejection Sampling

Target: mixture of 3 bivariate Gaussians. Proposal: broad 2D Gaussian.

In [ ]:
means = [np.array([-2, -2]), np.array([2, 2]), np.array([-1, 3])]
covs = [np.array([[0.5, 0.2], [0.2, 0.5]]),
        np.array([[0.8, -0.3], [-0.3, 0.4]]),
        np.array([[0.3, 0.0], [0.0, 0.6]])]
mix_weights = [0.4, 0.35, 0.25]

def target_pdf(x):
    return sum(w * stats.multivariate_normal.pdf(x, mean=mu, cov=cov)
               for w, mu, cov in zip(mix_weights, means, covs))

proposal_mean = np.array([0, 1])
proposal_cov = np.array([[6, 0], [0, 6]])

def proposal_pdf(x):
    return stats.multivariate_normal.pdf(x, mean=proposal_mean, cov=proposal_cov)

def proposal_sample():
    return np.random.multivariate_normal(proposal_mean, proposal_cov)

test_points = np.random.multivariate_normal(proposal_mean, proposal_cov, 10000)
ratios = np.array([target_pdf(x) / proposal_pdf(x) for x in test_points])
M = ratios.max() * 1.2
print(f'Scaling constant M = {M:.4f}')

In [ ]:
def rejection_sampling(n_samples, target, proposal_sample_fn, proposal_pdf_fn, M):
    samples, rejected_pts = [], []
    accepted, total = 0, 0
    while accepted < n_samples:
        x = proposal_sample_fn()
        u = np.random.uniform()
        total += 1
        if u <= target(x) / (M * proposal_pdf_fn(x)):
            samples.append(x)
            accepted += 1
        elif len(rejected_pts) < 2000:
            rejected_pts.append(x)
    return np.array(samples), np.array(rejected_pts), total

n_target = 3000
samples_rej, rejected, total_proposals = rejection_sampling(
    n_target, target_pdf, proposal_sample, proposal_pdf, M)
acc_rate = n_target / total_proposals
print(f'Acceptance rate: {acc_rate:.1%} ({total_proposals} proposals for {n_target} samples)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-4, 6, 100))
grid = np.column_stack([xx.ravel(), yy.ravel()])
zz = np.array([target_pdf(p) for p in grid]).reshape(xx.shape)

axes[0].contourf(xx, yy, zz, levels=20, cmap='Blues')
axes[0].set_title('Target Distribution', fontweight='bold')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')

if len(rejected) > 0:
    axes[1].scatter(rejected[:500, 0], rejected[:500, 1], s=5, alpha=0.3, c='gray', label='Rejected')
axes[1].scatter(samples_rej[:500, 0], samples_rej[:500, 1], s=8, alpha=0.5, c='#cd0000', label='Accepted')
axes[1].set_title(f'Rejection Sampling (rate={acc_rate:.1%})', fontweight='bold')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$'); axes[1].legend(fontsize=9)

axes[2].hist(samples_rej[:, 0], bins=50, density=True, alpha=0.6, color='#1a3a6e', label='Samples')
x1_range = np.linspace(-5, 5, 200)
marginal = np.array([np.trapz([target_pdf(np.array([x1, x2]))
                    for x2 in np.linspace(-4, 6, 100)], np.linspace(-4, 6, 100)) for x1 in x1_range])
axes[2].plot(x1_range, marginal, color='#cd0000', linewidth=2, label='True marginal')
axes[2].set_title('Marginal $x_1$', fontweight='bold')
axes[2].set_xlabel('$x_1$'); axes[2].set_ylabel('Density'); axes[2].legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'bishop_ch14_rejection')
plt.show()

## 2. Metropolis-Hastings Algorithm

In [ ]:
mu_target = np.array([1.0, -0.5])
cov_target = np.array([[1.0, 0.7], [0.7, 1.5]])

def log_target(x):
    diff = x - mu_target
    return -0.5 * diff @ np.linalg.inv(cov_target) @ diff

def metropolis_hastings(log_target, x0, n_samples, proposal_std=0.5):
    dim = len(x0)
    chain = np.zeros((n_samples, dim))
    chain[0] = x0
    current = x0.copy()
    current_log_p = log_target(current)
    accepted = 0
    for i in range(1, n_samples):
        proposal = current + np.random.normal(0, proposal_std, dim)
        proposal_log_p = log_target(proposal)
        if np.log(np.random.uniform()) < proposal_log_p - current_log_p:
            current = proposal
            current_log_p = proposal_log_p
            accepted += 1
        chain[i] = current
    return chain, accepted / (n_samples - 1)

x0 = np.array([0.0, 0.0])
n_mh = 10000
results = {}
for std in [0.1, 0.5, 2.0, 10.0]:
    chain, acc = metropolis_hastings(log_target, x0, n_mh, proposal_std=std)
    results[std] = (chain, acc)
    print(f'Proposal std={std:.1f}: acceptance rate={acc:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

xx, yy = np.meshgrid(np.linspace(-3, 5, 100), np.linspace(-4, 3, 100))
zz = stats.multivariate_normal.pdf(np.column_stack([xx.ravel(), yy.ravel()]),
                                    mean=mu_target, cov=cov_target).reshape(xx.shape)

for idx, (std, (chain, acc)) in enumerate(results.items()):
    ax = axes[idx]
    ax.contour(xx, yy, zz, levels=5, colors='gray', alpha=0.5)
    ax.plot(chain[:500, 0], chain[:500, 1], '-', color='#1a3a6e', alpha=0.3, linewidth=0.5)
    ax.scatter(chain[1000:, 0], chain[1000:, 1], s=3, alpha=0.1, c='#cd0000')
    ax.scatter(chain[0, 0], chain[0, 1], s=100, c='#228B22', marker='*', zorder=5)
    ax.set_title(f'$\\sigma_{{prop}}$={std}, accept={acc:.1%}')
    ax.set_xlim(-3, 5); ax.set_ylim(-4, 3)

fig.suptitle('Metropolis-Hastings: Effect of Proposal Scale', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch14_metropolis')
plt.show()

In [ ]:
# Trace plots and autocorrelation
best_chain = results[0.5][0]
burn_in = 1000
samples_mh = best_chain[burn_in:]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(best_chain[:, 0], color='#1a3a6e', linewidth=0.5, alpha=0.7)
axes[0, 0].axhline(mu_target[0], color='#cd0000', linestyle='--')
axes[0, 0].axvline(burn_in, color='gray', linestyle=':', alpha=0.5)
axes[0, 0].set_ylabel('$x_1$'); axes[0, 0].set_title('Trace Plot $x_1$', fontweight='bold')

axes[0, 1].plot(best_chain[:, 1], color='#228B22', linewidth=0.5, alpha=0.7)
axes[0, 1].axhline(mu_target[1], color='#cd0000', linestyle='--')
axes[0, 1].axvline(burn_in, color='gray', linestyle=':', alpha=0.5)
axes[0, 1].set_ylabel('$x_2$'); axes[0, 1].set_title('Trace Plot $x_2$', fontweight='bold')

for dim, color, ax in [(0, '#1a3a6e', axes[1, 0]), (1, '#228B22', axes[1, 1])]:
    max_lag = 100
    x = samples_mh[:, dim]
    x_c = x - x.mean()
    autocorr = np.correlate(x_c, x_c, mode='full')
    autocorr = autocorr[len(autocorr)//2:max_lag+1+len(autocorr)//2]
    autocorr = autocorr[:max_lag+1] / autocorr[0]
    ax.bar(range(len(autocorr)), autocorr, color=color, alpha=0.7)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('Lag'); ax.set_ylabel('ACF')
    ax.set_title(f'Autocorrelation $x_{dim+1}$', fontweight='bold')

fig.tight_layout()
plt.show()

print(f'Sample mean: {samples_mh.mean(axis=0).round(4)}')
print(f'True mean:   {mu_target}')
print(f'Sample cov:\n{np.cov(samples_mh.T).round(4)}')
print(f'True cov:\n{cov_target}')

## 3. Gibbs Sampling

Conditional distributions for bivariate Gaussian:
$$x_1 | x_2 \sim \mathcal{N}\left(\mu_1 + \rho\frac{\sigma_1}{\sigma_2}(x_2 - \mu_2),\; \sigma_1^2(1-\rho^2)\right)$$

In [ ]:
def gibbs_bivariate_gaussian(mu, cov, x0, n_samples):
    sigma1 = np.sqrt(cov[0, 0])
    sigma2 = np.sqrt(cov[1, 1])
    rho = cov[0, 1] / (sigma1 * sigma2)
    chain = np.zeros((n_samples, 2))
    chain[0] = x0
    x = x0.copy()
    for i in range(1, n_samples):
        x[0] = np.random.normal(mu[0] + rho*(sigma1/sigma2)*(x[1]-mu[1]),
                                sigma1*np.sqrt(1-rho**2))
        x[1] = np.random.normal(mu[1] + rho*(sigma2/sigma1)*(x[0]-mu[0]),
                                sigma2*np.sqrt(1-rho**2))
        chain[i] = x.copy()
    return chain

mu_gibbs = np.array([2.0, -1.0])
cov_gibbs = np.array([[1.0, 0.8], [0.8, 1.0]])
x0_gibbs = np.array([-3.0, 3.0])
gibbs_chain = gibbs_bivariate_gaussian(mu_gibbs, cov_gibbs, x0_gibbs, 5000)
print(f'Gibbs chain: {gibbs_chain.shape}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

xx_g, yy_g = np.meshgrid(np.linspace(-2, 6, 100), np.linspace(-5, 3, 100))
zz_g = stats.multivariate_normal.pdf(
    np.column_stack([xx_g.ravel(), yy_g.ravel()]),
    mean=mu_gibbs, cov=cov_gibbs).reshape(xx_g.shape)

n_show = 50
axes[0].contour(xx_g, yy_g, zz_g, levels=5, colors='gray', alpha=0.5)
for i in range(n_show - 1):
    axes[0].plot([gibbs_chain[i,0], gibbs_chain[i+1,0]],
                 [gibbs_chain[i,1], gibbs_chain[i,1]], 'b-', alpha=0.4, lw=0.8)
    axes[0].plot([gibbs_chain[i+1,0], gibbs_chain[i+1,0]],
                 [gibbs_chain[i,1], gibbs_chain[i+1,1]], 'r-', alpha=0.4, lw=0.8)
axes[0].scatter(gibbs_chain[:n_show,0], gibbs_chain[:n_show,1], s=15,
                c=range(n_show), cmap='viridis', zorder=5)
axes[0].set_title(f'Gibbs: First {n_show} Steps', fontweight='bold')

burn = 500
axes[1].contour(xx_g, yy_g, zz_g, levels=5, colors='gray', alpha=0.5)
axes[1].scatter(gibbs_chain[burn:,0], gibbs_chain[burn:,1], s=3, alpha=0.2, c='#cd0000')
axes[1].set_title(f'Gibbs Samples (burn-in={burn})', fontweight='bold')

gibbs_post = gibbs_chain[burn:]
axes[2].hist(gibbs_post[:,0], bins=40, density=True, alpha=0.5, color='#1a3a6e', label='$x_1$')
axes[2].hist(gibbs_post[:,1], bins=40, density=True, alpha=0.5, color='#cd0000', label='$x_2$')
xr = np.linspace(-2, 6, 200)
axes[2].plot(xr, stats.norm.pdf(xr, mu_gibbs[0], 1), '#1a3a6e', lw=2, ls='--')
axes[2].plot(xr, stats.norm.pdf(xr, mu_gibbs[1], 1), '#cd0000', lw=2, ls='--')
axes[2].set_title('Marginal Distributions', fontweight='bold')
axes[2].legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'bishop_ch14_gibbs')
plt.show()

## 4. Effect of Correlation on Gibbs Efficiency

In [ ]:
rhos = [0.0, 0.5, 0.9, 0.99]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx, rho in enumerate(rhos):
    cov_test = np.array([[1.0, rho], [rho, 1.0]])
    chain_test = gibbs_bivariate_gaussian(np.zeros(2), cov_test, np.zeros(2), 3000)
    axes[idx].plot(chain_test[:200, 0], linewidth=0.8, color='#1a3a6e')
    axes[idx].set_title(f'$\\rho$ = {rho}')
    axes[idx].set_xlabel('Iteration'); axes[idx].set_ylabel('$x_1$')
    x = chain_test[500:, 0]
    x_c = x - x.mean()
    acf1 = np.correlate(x_c[:-1], x_c[1:])[0] / np.correlate(x_c, x_c)[0]
    ess = len(x) * (1 - acf1) / (1 + acf1)
    axes[idx].text(0.95, 0.95, f'ESS$\\approx${ess:.0f}',
                   transform=axes[idx].transAxes, ha='right', va='top', fontsize=9)

fig.suptitle('Gibbs Sampling: Mixing vs Correlation', fontsize=14)
fig.tight_layout()
plt.show()

print(f'Sample mean: {gibbs_post.mean(axis=0).round(4)}')
print(f'True mean:   {mu_gibbs}')
print(f'Sample cov:\n{np.cov(gibbs_post.T).round(4)}')

## 5. Importance Sampling

In [ ]:
# Estimate E[f(x)] under target using importance sampling
def importance_sampling(f, target, proposal_sample_fn, proposal_pdf_fn, n_samples):
    xs = np.array([proposal_sample_fn() for _ in range(n_samples)])
    w = np.array([target(x) / proposal_pdf_fn(x) for x in xs])
    w_norm = w / w.sum()
    return np.sum(w_norm * np.array([f(x) for x in xs])), w_norm

f_test = lambda x: x[0]  # E[x1]
est_mean, is_weights = importance_sampling(f_test, target_pdf, proposal_sample, proposal_pdf, 5000)

true_mean_x1 = sum(w * mu[0] for w, mu in zip(mix_weights, means))
print(f'IS estimate of E[x1]: {est_mean:.4f}')
print(f'True E[x1]:           {true_mean_x1:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(is_weights, bins=50, color='#1a3a6e', alpha=0.7, edgecolor='white')
ax.set_xlabel('Normalized Weight')
ax.set_ylabel('Count')
ax.set_title('Importance Sampling Weight Distribution', fontweight='bold')
eff_ss = 1.0 / np.sum(is_weights**2)
ax.text(0.95, 0.95, f'Effective SS = {eff_ss:.0f}/{len(is_weights)}',
        transform=ax.transAxes, ha='right', va='top', fontsize=11)
plt.tight_layout()
save_fig(fig, 'bishop_ch14_importance')
plt.show()

## 6. Hamiltonian Monte Carlo (HMC)

In [ ]:
def hmc(log_prob, grad_log_prob, x0, n_samples, step_size=0.1, n_leapfrog=20):
    dim = len(x0)
    chain = np.zeros((n_samples, dim))
    chain[0] = x0
    current = x0.copy()
    accepted = 0

    for i in range(1, n_samples):
        p = np.random.randn(dim)
        q = current.copy()
        p_current = p.copy()

        # Leapfrog integration
        p += 0.5 * step_size * grad_log_prob(q)
        for _ in range(n_leapfrog - 1):
            q += step_size * p
            p += step_size * grad_log_prob(q)
        q += step_size * p
        p += 0.5 * step_size * grad_log_prob(q)
        p = -p

        current_H = -log_prob(current) + 0.5 * np.sum(p_current**2)
        proposed_H = -log_prob(q) + 0.5 * np.sum(p**2)

        if np.log(np.random.uniform()) < current_H - proposed_H:
            current = q
            accepted += 1
        chain[i] = current

    return chain, accepted / (n_samples - 1)

cov_inv = np.linalg.inv(cov_target)
grad_log = lambda x: -cov_inv @ (x - mu_target)

hmc_chain, hmc_acc = hmc(log_target, grad_log, np.zeros(2), 5000, step_size=0.15, n_leapfrog=15)
print(f'HMC acceptance rate: {hmc_acc:.3f}')
print(f'HMC sample mean: {hmc_chain[500:].mean(axis=0).round(4)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].contour(xx, yy, zz, levels=5, colors='gray', alpha=0.5)
axes[0].scatter(hmc_chain[500:,0], hmc_chain[500:,1], s=3, alpha=0.15, c='#228B22')
axes[0].set_title(f'HMC Samples (acc={hmc_acc:.1%})', fontweight='bold')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')

mh_chain = results[0.5][0]
axes[1].contour(xx, yy, zz, levels=5, colors='gray', alpha=0.5)
axes[1].scatter(mh_chain[500:,0], mh_chain[500:,1], s=3, alpha=0.15, c='#cd0000')
axes[1].set_title(f'MH Samples (acc={results[0.5][1]:.1%})', fontweight='bold')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')

fig.suptitle('HMC vs Metropolis-Hastings', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch14_hmc_vs_mh')
plt.show()

## Summary

- **Rejection sampling**: exact but inefficient in high dimensions
- **Metropolis-Hastings**: flexible MCMC with tunable proposal scale
- **Gibbs sampling**: component-wise updates using conditionals
- **Importance sampling**: weighted reuse of proposal samples
- **HMC**: uses gradient information for efficient exploration

In [ ]:
takeaways = [
    '1. Rejection sampling acceptance rate drops exponentially with dimension.',
    '2. MH acceptance rate depends critically on proposal scale (~23% optimal in high-d).',
    '3. Gibbs sampling avoids tuning a proposal but slows with high correlations.',
    '4. Importance sampling can estimate expectations without MCMC.',
    '5. HMC exploits gradients for much better mixing than random-walk MH.'
]
print('Key Takeaways:')
for t in takeaways:
    print(t)